In [1]:
import gradio as gr
gr.close_all()  # kill any zombie servers from previous runs

import os
import base64
import io
from dotenv import load_dotenv
from openai import OpenAI
from groq import Groq
from PIL import Image

In [2]:
load_dotenv()

DEEPSEEK_KEY = os.getenv("DEEPSEEK_API_KEY")
GROQ_KEY = os.getenv("GROQ_API_KEY")
OPENROUTER_KEY = os.getenv("OPENROUTER_API_KEY")

deepseek_client = OpenAI(api_key=DEEPSEEK_KEY, base_url="https://api.deepseek.com/v1")
groq_client = Groq(api_key=GROQ_KEY)
openrouter_client = OpenAI(api_key=OPENROUTER_KEY, base_url="https://openrouter.ai/api/v1")

VISION_MODEL = "meta-llama/llama-3.2-11b-vision-instruct"

In [3]:
CONSULTANT_PROMPT = """You are BLVCK N GREENS PHARMA, a professional medical consultant AI.
Give evidence-based health advice, home remedies, general OTC guidance, and refer to a doctor when needed.
Always consider the patient's profile below. Never diagnose definitively — always add a short disclaimer.

PATIENT PROFILE:
{profile}

Format answers in clean markdown with sections like **Assessment**, **Home Remedies**, **When to See a Doctor**."""

GUIDE_PROMPT = """You are BLVCK N GREENS PHARMA in GUIDE mode.
Provide step-by-step health guidance, wellness tips and lifestyle advice.

PATIENT PROFILE:
{profile}"""

FEEDBACK_PROMPT = """You are BLVCK N GREENS PHARMA in FEEDBACK mode.
Give constructive, honest feedback on health habits, symptoms or concerns.

PATIENT PROFILE:
{profile}"""

VISION_PROMPT = """You are a dermatology-focused AI for BLVCK N GREENS PHARMA.
Analyze the uploaded image of skin or physical change. Report:
1. **Observation** – color, texture, shape, distribution.
2. **Possible Conditions** – 2-4 likely possibilities with brief notes.
3. **Urgency Level** – Low / Moderate / High.
4. **Recommended Action** – home care + when to see a dermatologist.
Do NOT give a definitive diagnosis. Always recommend professional confirmation."""


def build_profile(height, weight, sex, age, genotype, conditions, allergies, meds):
    return f"""- Height: {height or 'N/A'} cm
- Weight: {weight or 'N/A'} kg
- Sex: {sex or 'N/A'}
- Age: {age or 'N/A'}
- Genotype: {genotype or 'N/A'}
- Existing Conditions: {conditions or 'None'}
- Allergies: {allergies or 'None'}
- Current Medications: {meds or 'None'}"""


def ask_deepseek(messages):
    try:
        r = deepseek_client.chat.completions.create(
            model="deepseek-chat",
            messages=messages,
            temperature=0.6,
            max_tokens=1200,
        )
        return r.choices[0].message.content
    except Exception as e:
        return f"⚠️ DeepSeek error: {e}"


def ask_groq(messages):
    try:
        r = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages,
            temperature=0.6,
            max_tokens=1200,
        )
        return r.choices[0].message.content
    except Exception as e:
        return f"⚠️ Groq error: {e}"


def analyze_image_groq(image: Image.Image, user_note: str = ""):
    """Vision analysis via Groq with automatic model fallback."""
    if image is None:
        return "Please upload an image."

    buffered = io.BytesIO()
    image.save(buffered, format="PNG")
    b64 = base64.b64encode(buffered.getvalue()).decode()

    vision_models = [
        "qwen/qwen3.6-27b",
        "qwen/qwen3.8-27b",
        "meta-llama/llama-4-scout-17b-16e-instruct",
        "meta-llama/llama-4-maverick-17b-128e-instruct",
    ]

    last_error = None
    for model_id in vision_models:
        try:
            r = groq_client.chat.completions.create(
                model=model_id,
                messages=[{
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": f"{VISION_PROMPT}\n\n{user_note or 'Analyze this image.'}",
                        },
                        {
                            "type": "image_url",
                            "image_url": {"url": f"data:image/png;base64,{b64}"},
                        },
                    ],
                }],
                temperature=0.4,
                max_tokens=1000,
            )
            return r.choices[0].message.content
        except Exception as e:
            last_error = f"{model_id}: {e}"
            continue

    return f"⚠️ Groq vision error — all models failed.\nLast error: {last_error}"


def chat_fn(message, history, mode, height, weight, sex, age, genotype,
            conditions, allergies, meds, image):
    profile = build_profile(height, weight, sex, age, genotype, conditions, allergies, meds)
    history = history or []

    if image is not None:
        note = message.strip() if message.strip() else "Analyze this skin / body change."
        result = analyze_image_groq(image, note)
        return history + [
            {"role": "user", "content": note},
            {"role": "assistant", "content": result},
        ], None, ""

    if mode == "Consult (DeepSeek)":
        sys_prompt = CONSULTANT_PROMPT.format(profile=profile)
        llm = ask_deepseek
    elif mode == "Guide (Groq)":
        sys_prompt = GUIDE_PROMPT.format(profile=profile)
        llm = ask_groq
    else:
        sys_prompt = FEEDBACK_PROMPT.format(profile=profile)
        llm = ask_groq

    messages = [{"role": "system", "content": sys_prompt}]
    for turn in history:
        role = turn.get("role")
        content = turn.get("content")
        if role in ("user", "assistant") and isinstance(content, str):
            messages.append({"role": role, "content": content})
    messages.append({"role": "user", "content": message})

    reply = llm(messages)
    return history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": reply},
    ], None, ""

In [4]:
theme = gr.themes.Base(
    primary_hue=gr.themes.colors.green,
    secondary_hue=gr.themes.colors.green,
    neutral_hue=gr.themes.colors.gray,
    font=[gr.themes.GoogleFont("Inter"), "sans-serif"],
).set(
    body_background_fill="#0a0a0a",
    body_background_fill_dark="#0a0a0a",
    block_background_fill="#111111",
    block_border_color="#1f7a1f",
    block_label_text_color="#7CFC00",
    block_title_text_color="#7CFC00",
    body_text_color="#e8ffe8",
    input_background_fill="#0f1a0f",
    button_primary_background_fill="#1f7a1f",
    button_primary_background_fill_hover="#2fa32f",
    button_primary_text_color="#ffffff",
)

CSS = """
body, .gradio-container { background: #0a0a0a !important; }
#title { text-align:center; padding: 18px; }
#title h1 { color: #7CFC00; font-size: 32px; font-weight: 800;
  letter-spacing: 2px; margin-bottom: 4px; }
#title p { color: #9be89b; margin-top: 0; }
.gradio-container .prose h1, .gradio-container .prose h2,
.gradio-container .prose h3 { color:#7CFC00; }
footer { display:none !important; }

/* Hide the mode radio completely */
#mode_selector { display: none !important; }

/* ---- Chatbot message bubbles ---- */
.gradio-container .message-wrap,
.gradio-container .message {
    background: #0f1a0f !important;
    color: #e8ffe8 !important;
    border: 1px solid #1f7a1f !important;
}
.gradio-container .message p,
.gradio-container .message li,
.gradio-container .message span,
.gradio-container .message strong,
.gradio-container .message em,
.gradio-container .message h1,
.gradio-container .message h2,
.gradio-container .message h3,
.gradio-container .message h4 {
    color: #e8ffe8 !important;
}
.gradio-container .message.user,
.gradio-container .user > .message {
    background: #14331a !important;
    border-color: #2fa32f !important;
}
.gradio-container .message.bot,
.gradio-container .bot > .message {
    background: #0f1a0f !important;
    border-color: #1f7a1f !important;
}
.gradio-container .chatbot,
.gradio-container .chatbot .panel {
    background: #0a0a0a !important;
    border: 1px solid #1f7a1f !important;
}
.gradio-container .message code,
.gradio-container .message pre {
    background: #071207 !important;
    color: #7CFC00 !important;
}
"""

with gr.Blocks(title="BLVCK N GREENS PHARMA") as demo:
    gr.HTML("""
    <div id="title">
      <h1>BLVCK N GREENS PHARMA</h1>
      <p>AI Powered Medical Consultant • Skin & Body Change Analyzer</p>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 🧬 Patient Profile")
            height = gr.Number(label="Height (cm)", value=170)
            weight = gr.Number(label="Weight (kg)", value=70)
            sex = gr.Dropdown(["Male", "Female", "Other"], label="Sex", value="Male")
            age = gr.Number(label="Age", value=25)
            genotype = gr.Dropdown(
                ["AA", "AS", "SS", "AC", "SC", "CC", "Unknown"],
                label="Genotype", value="AA",
            )
            conditions = gr.Textbox(label="Existing Illness / Health Conditions",
                                    placeholder="e.g. Hypertension, Asthma", lines=2)
            allergies = gr.Textbox(label="Allergies", placeholder="e.g. Penicillin", lines=2)
            meds = gr.Textbox(label="Current Medications", placeholder="e.g. Metformin 500mg", lines=2)

            # Hidden mode selector — defaults to Consult (DeepSeek)
            with gr.Group(elem_id="mode_selector"):
                mode = gr.Radio(
                    ["Consult (DeepSeek)", "Guide (Groq)", "Feedback (Groq)"],
                    label="Mode", value="Consult (DeepSeek)",
                )

            gr.Markdown("### 🔬 Image Analysis")
            image_in = gr.Image(label="Upload skin / body change", type="pil")

        with gr.Column(scale=2):
            chatbot = gr.Chatbot(height=560,
                                 label="BLVCK N GREENS PHARMA Assistant",
                                 avatar_images=(None, None))
            with gr.Row():
                msg = gr.Textbox(placeholder="Describe your symptoms...",
                                 scale=8, container=False)
                send = gr.Button("Send", variant="primary", scale=1)
            clear = gr.Button("Clear Chat")

    inputs = [msg, chatbot, mode, height, weight, sex, age, genotype,
              conditions, allergies, meds, image_in]

    send.click(chat_fn, inputs, [chatbot, image_in, msg])
    msg.submit(chat_fn, inputs, [chatbot, image_in, msg])
    clear.click(lambda: ([], None, ""), None, [chatbot, image_in, msg], queue=False)

In [5]:
import socket

def find_free_port(start=7860, end=7900):
    for p in range(start, end):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(("127.0.0.1", p)) != 0:
                return p
    return 0  # let Gradio decide

PORT = find_free_port()
print(f"Launching on port {PORT}")

demo.launch(
    server_name="127.0.0.1",
    server_port=PORT,
    show_error=True,
    theme=theme,
    css=CSS,
    inbrowser=True,        # auto-opens browser tab
    share=False,           # set True if you want a public gradio.live link
)

Launching on port 7863
* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
